# Éditeur mobile → Kokoro → MP4

Exporter votre projet depuis le site, puis exécuter ces cellules dans Colab depuis Chrome Android. Session CPU Python 3.12. Aucun jeton API payant. Le modèle gratuit et Chromium sont téléchargés sur la machine Colab. Les ressources gratuites et la durée de session dépendent de Google.


In [ ]:
import os, subprocess, sys, json
from pathlib import Path
assert sys.version_info[:2] == (3, 12), "Sélectionner Python 3.12"
REF = "feat/mobile-json-editor"  # Après fusion de la PR, main peut être utilisé.
PROJECT = Path('/content/Destiny-Explained-Studio')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch','https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(PROJECT)],check=True)
subprocess.run(['git','-C',str(PROJECT),'rev-parse','HEAD'],check=True)


In [ ]:
%%bash
set -euo pipefail
cd /content
NODE_VERSION=22.23.4
NODE_ARCHIVE="node-v${NODE_VERSION}-linux-x64.tar.xz"
curl -fsSLO "https://nodejs.org/dist/v${NODE_VERSION}/${NODE_ARCHIVE}"
curl -fsSLo node-SHASUMS256.txt "https://nodejs.org/dist/v${NODE_VERSION}/SHASUMS256.txt"
awk -v f="$NODE_ARCHIVE" '$2 == f' node-SHASUMS256.txt | sha256sum --check -
tar -xJf "$NODE_ARCHIVE"
apt-get update -qq
AUDIO_PACKAGE=libasound2
if apt-cache show libasound2t64 >/dev/null 2>&1; then AUDIO_PACKAGE=libasound2t64; fi
apt-get install -y ffmpeg python3-venv libnss3 libatk-bridge2.0-0 libx11-xcb1 libxcomposite1 libxdamage1 libxrandr2 libgbm1 "$AUDIO_PACKAGE" fonts-liberation
python3 -m venv /content/destiny-tts
/content/destiny-tts/bin/pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cpu
/content/destiny-tts/bin/pip install -r /content/Destiny-Explained-Studio/remotion/tts/requirements-kokoro.txt


## Importer le JSON exporté
Choisir un seul fichier JSON dans les téléchargements Android. L’original reste dans `out/editor-upload.json`. La copie pour rendu écarte les champs inconnus : le rapport les liste. Les erreurs de schéma et de durée bloquent le traitement. Aucun exemple du dépôt n’est modifié.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, "Choisir exactement un fichier JSON"
name, data = next(iter(uploaded.items()))
assert name.lower().endswith('.json') and len(data) <= 2*1024*1024, "JSON de 2 Mo maximum"
ROOT = PROJECT / 'remotion'
(ROOT / 'out').mkdir(exist_ok=True)
(ROOT / 'out/editor-upload.json').write_bytes(data)


In [ ]:
%%bash
set -euo pipefail
export PATH="/content/destiny-tts/bin:/content/node-v22.23.4-linux-x64/bin:$PATH"
export HF_HUB_DISABLE_XET=1
cd /content/Destiny-Explained-Studio/remotion
npm ci
npm run typecheck
node --experimental-strip-types scripts/import-editor.mjs out/editor-upload.json --compatible-copy
python scripts/narration.py out/editor-episode.json
npm run browser:ensure
npm run render:episode -- out/editor-episode-ready.json


## Télécharger le MP4 et les rapports
Kokoro utilise af_heart à 1× par défaut. Une narration trop longue échoue explicitement ; raccourcir le texte ou allonger la scène dans l’éditeur puis réimporter. Les ancrages de phrases sont contrôlés avec les mots réellement prononcés. Enregistrer les résultats avant la fin de la session Colab.


In [ ]:
ready = json.loads((ROOT / 'out/editor-episode-ready.json').read_text())
video = (ROOT / ready['render']['output']).resolve()
assert video.is_relative_to(ROOT.resolve()) and video.suffix.lower() == '.mp4'
assert video.exists(), "Rendu MP4 absent"
files.download(str(video))
files.download(str(ROOT / 'out/editor-import-report.json'))
files.download(str(ROOT / 'out/editor-episode-ready.json'))
